In [2]:
import numpy as np
from matplotlib import pyplot as plt
import netCDF4 as nc
from cmocean import cm
from scipy import signal
import seawater as sw
import geopy
import geopy.distance
#from eofs.standard import Eof
import datetime
from datetime import date
from scipy import interpolate
from os import listdir
from os.path import isfile, join
import xarray as xr
from sklearn.decomposition import PCA
import cartopy
import cartopy.crs as ccrs
import matplotlib.patches as mpatches
import cartopy.mpl.ticker as cticker
from shapely.geometry import Point, Polygon
from calendar import monthrange
from scipy import stats

GS_lat1 = 30.125; GS_lat2 = 48.875;
GS_lon1 = 280.125; GS_lon2 = 309.875;

def cal_path_position(path,LON1,LON2 #track path
    ####path:(ntimes,x,2)
    ###LON1 and LON2 are boundaries for region of interest with given lon boundaries
    times = path.shape[0]
    path_position = np.zeros(times)
    for k in range(times):
        path_lon = path[k,:,0][np.where(np.isnan(path[k,:,0])==0)]
        path_lat = path[k,:,1][np.where(np.isnan(path[k,:,1])==0)]
        ind = np.where((path_lon>=LON1)&(path_lon<=LON2))[0]
        path_position[k] = np.mean(path_lat[ind[0]:ind[-1]+1])     
    return path_position

def cal_path_length(path,LON1,LON2): #length of gulfstream
    ####path:(ntimes,x,2)
    ###LON1 and LON2 are boundaries for region of interest
    times = path.shape[0]
    path_length = np.zeros(times)
    for k in range(times):
        path_lon = path[k,:,0][np.where(np.isnan(path[k,:,0])==0)]
        path_lat = path[k,:,1][np.where(np.isnan(path[k,:,1])==0)]
        ind = np.where((path_lon>=LON1)&(path_lon<=LON2))[0]
        path_lon = path_lon[ind[0]:ind[-1]+1]
        path_lat = path_lat[ind[0]:ind[-1]+1]
        dist1 = 0;
        for p in range(path_lon.size-1):
            x1 = path_lon[p]; x2 = path_lon[p+1];
            y1 = path_lat[p]; y2 = path_lat[p+1];
            dist1 = dist1+sw.dist([y1,y2],[x1,x2])[0][0];
        path_length[k] = dist1;
    return path_length

def month2year(series):
    ##series(nmonths)
    nmonths = series.shape[0]
    nyears = int(nmonths/12)
    new = np.zeros(nyears)
    for n in range(nyears):
        new[n] = np.mean(series[n*12:(n+1)*12])
    return new

def month2year3d(data):
    ##data(nmonths,nlat,nlon)
    nmonths = data.shape[0]
    nyears = int(nmonths/12)
    new = np.zeros((nyears,data.shape[1],data.shape[2]))
    for n in range(nyears):
        new[n,:,:] = np.mean(data[n*12:(n+1)*12,:,:],axis=0)
    return new


In [3]:
year1 = 1993; year2 = 2018;
lat0 = np.arange(-89.875,89.875+0.25,0.25,dtype='float32');
lon0 = np.arange(0.125,359.875+0.25,0.25,dtype='float32');
ilat_obs = np.where((lat0>=GS_lat1) & (lat0<=GS_lat2))[0];
ilon_obs = np.where((lon0>=GS_lon1) & (lon0<=GS_lon2))[0];
ssh_dataset = nc.Dataset('/Volumes/Yiming91977/Global_SSH_AVISO/Global_MONTHLY_SSH_1993_2020.nc','r')
ssh_obs = ssh_dataset.variables['ssh'][0:(year2-year1+1)*12,ilat_obs,ilon_obs];
ssh_obs[np.where(abs(ssh_obs)>10)] = np.nan
ssh_obs_mean = np.mean(ssh_obs,axis=0)
lat_obs = ssh_dataset.variables['lat'][ilat_obs];
lon_obs = ssh_dataset.variables['lon'][ilon_obs];
nyears_obs = int(year2-year1+1)
nmonths_obs = ssh_obs.shape[0];
path_obs = np.nan*np.zeros((ssh_obs.shape[0],400,2));   ####320 is for the longest path
ax_obs = 0.25
for k in range(ssh_obs.shape[0]):
    c = plt.contour(lon_obs,lat_obs,ssh_obs[k,:,:],levels=[ax_obs])
    seg = c.collections[0].get_paths()
    length = np.zeros(len(seg))
    for p in range(len(seg)):
        length[p] = seg[p].vertices.shape[0];
    real = np.where(length==max(length))[0][0]   ####choose the real path
    path1 = c.collections[0].get_paths()[real].vertices
    path_obs[k,0:path1.shape[0],:] = path1
    plt.close();

SI_obs = cal_path_position(path_obs,LON1=285,LON2=290)
SIa_obs = month2year((SI_obs-np.mean(SI_obs))/np.std(SI_obs));
PI_obs = cal_path_position(path_obs,LON1=lon_obs[0],LON2=lon_obs[-1])
PIa_obs = month2year((PI_obs-np.mean(PI_obs))/np.std(PI_obs));

length_obs = cal_path_length(path_obs,LON1=lon_obs[0],LON2=lon_obs[-1])
length_obs = month2year((length_obs-np.mean(length_obs))/np.std(length_obs))


In [1]:
ssh = ds['ssh_glorys'[1,0,:,:]
lat_roms = ds['lat_roms'][:]
lon_roms = ds['lon_roms'][:]
plt.contourf(lon,lat, ssh, np.linspace(-1,1,21), cmap= 'seismic', extend = 'both');plt.colorbar();
#narrow white band is the gulf stream

SyntaxError: '[' was never closed (1394549574.py, line 1)

In [2]:
ax_obs = 0.25 # threshold of ssh level representing gulfstream 
for k in range(ssh_obs.shape[0]):
c = plt.contour(lon_obs,lat_obs,ssh_obs[k,:,:],levels=[ax_obs]) # it will only track a single contour line 
seg = c.collections[0].get_paths() # length is 4, there are 4 contour lines but for culfstream we need to identify 1. 
#seg[0] returns lon and lat of the contour lines. given multiple segs, identify gulfstream: gulf stream is long, all the segments the longest one is prob the gulf stream. in his paper he uses -75 and -70 lon. 
length = np.zeros(len(seg))
for p in range(len(seg)):
    length[p] = seg[p].vertices.shape[0 #extract the length len(seg[p])
real = np.where(length==max(length))[0][0]   ####choose the real path
path1 = c.collections[0].get_paths()[real].vertices
path_obs[k,0:path1.shape[0],:] = path1
plt.close();
#double check that path1 is overlaping with the contour map and seeing how good. changing the threshold .25,.20,.30 and seeing where is the best representation. want it to be int he middle of the contour path 

IndentationError: expected an indented block after 'for' statement on line 2 (1123258488.py, line 3)